In [7]:
import cv2
import numpy as np
import os

In [8]:
# config
VIDEO_DIR= r"d:\Duy Toan\Python\DUT AI Club\Homework\day 28 3-3-2026"
number_of_batch_frame = 6
path_to_video= os.path.join(VIDEO_DIR, "dataset.mp4")

print("Video path :", path_to_video)
print("File exists:", os.path.exists(path_to_video))

Video path : d:\Duy Toan\Python\DUT AI Club\Homework\day 28 3-3-2026\dataset.mp4
File exists: True


In [9]:
# Bước 1 — Đọc video, kiểm tra frame
capture = cv2.VideoCapture(path_to_video)
ret, frame = capture.read()
capture.release()

print("ret   :", ret)           # True nếu đọc thành công
print("type  :", type(frame))
print("shape :", frame.shape)   # (height, width, 3)
print("dtype :", frame.dtype)   # uint8
print(f"  height={frame.shape[0]}, width={frame.shape[1]}, channels={frame.shape[2]}")

ret   : True
type  : <class 'numpy.ndarray'>
shape : (480, 640, 3)
dtype : uint8
  height=480, width=640, channels=3


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BASE CLASS — toàn bộ pipeline chung
# ═══════════════════════════════════════════════════════════════════════════════

class BackgroundSubtractionProcess:
    """
    Base class chứa full pipeline:
      read video → init background → loop(subtract → mask → count crossing → draw → write)

    Subclass chỉ cần override:
      _init_background(cap)  → (background, frames_buffer)
      _update_background(bg, buffer, new_gray)  → new background
      [optional] _get_mask(frame, background)   → (mask, gray)   (MOG override)
    """

    def __init__(self, video_path, n_batch=6, threshold=30, min_area=1000, line_ratio=0.5):
        self.video_path  = video_path
        self.n_batch     = n_batch       # số frame để tính background ban đầu
        self.threshold   = threshold     # ngưỡng binary
        self.min_area    = min_area      # diện tích contour tối thiểu (px²)
        self.line_ratio  = line_ratio    # vị trí vạch đếm (0.0 – 1.0, mặc định giữa)

    # ── I/O ──────────────────────────────────────────────────────────────────

    def _open_cap_writer(self, method_name):
        cap     = cv2.VideoCapture(self.video_path)
        w       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps     = cap.get(cv2.CAP_PROP_FPS) or 25
        line_y  = int(h * self.line_ratio)
        out_dir = os.path.dirname(os.path.abspath(self.video_path))
        out_path= os.path.join(out_dir, f"output_{method_name}.mp4")
        fourcc  = cv2.VideoWriter_fourcc(*"mp4v")
        writer  = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
        return cap, writer, out_path, line_y

    # ── Background (override trong subclass) ─────────────────────────────────

    def _init_background(self, cap):
        """Đọc batch frame đầu, trả về (background uint8, frames_buffer)."""
        raise NotImplementedError

    def _update_background(self, background, frames_buffer, new_gray):
        """Cập nhật background sau mỗi frame. Mặc định: tĩnh (không đổi)."""
        return background

    # ── Foreground mask (override trong MOG) ─────────────────────────────────

    def _get_mask(self, frame, background):
        """absdiff → threshold → dilate → erode. Trả về (mask, gray)."""
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(gray, background)
        _, mask = cv2.threshold(diff, self.threshold, 255, cv2.THRESH_BINARY)
        mask = cv2.dilate(mask, None, iterations=2)
        mask = cv2.erode(mask, None, iterations=2)
        return mask, gray

    # ── Contour → centroid ────────────────────────────────────────────────────

    def _get_centroids(self, mask):
        """Tìm contour đủ lớn, trả về list (cx, cy, x, y, w, h)."""
        contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        result = []
        for c in contours:
            if cv2.contourArea(c) < self.min_area:
                continue
            x, y, w, h = cv2.boundingRect(c)
            result.append((x + w // 2, y + h // 2, x, y, w, h))
        return result

    # ── Đếm người qua vạch ───────────────────────────────────────────────────

    def _update_counts(self, centroids, prev_centroids, line_y, count_in, count_out):
        """
        So sánh centroid hiện tại với frame trước (nearest-neighbor matching, dist < 80px).
        Đi từ trên xuống dưới qua line_y → IN
        Đi từ dưới lên trên qua line_y  → OUT
        """
        for (cx, cy, *_) in centroids:
            for (px, py, *_) in prev_centroids:
                if ((cx - px)**2 + (cy - py)**2) ** 0.5 < 80:
                    if py < line_y <= cy:
                        count_in  += 1
                    elif py >= line_y > cy:
                        count_out += 1
                    break
        return count_in, count_out

    # ── Vẽ lên frame ─────────────────────────────────────────────────────────

    def _draw(self, frame, centroids, line_y, count_in, count_out):
        h, w = frame.shape[:2]
        # Vạch ngưỡng đỏ giữa màn hình
        cv2.line(frame, (0, line_y), (w, line_y), (0, 0, 255), 2)
        cv2.putText(frame, "COUNTING LINE", (w // 2 - 80, line_y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        # Counter
        cv2.rectangle(frame, (0, 0), (180, 80), (0, 0, 0), -1)
        cv2.putText(frame, f"IN : {count_in}",  (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(frame, f"OUT: {count_out}", (10, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 100, 255), 2)
        # Bounding box + centroid
        for (cx, cy, x, y, bw, bh) in centroids:
            cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)
        return frame

    # ── Main loop ─────────────────────────────────────────────────────────────

    def run(self, method_name):
        cap, writer, out_path, line_y = self._open_cap_writer(method_name)
        background, frames_buffer     = self._init_background(cap)

        count_in, count_out = 0, 0
        prev_centroids      = []
        processed           = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            mask, gray   = self._get_mask(frame, background)
            centroids    = self._get_centroids(mask)
            count_in, count_out = self._update_counts(
                centroids, prev_centroids, line_y, count_in, count_out)

            annotated = self._draw(frame.copy(), centroids, line_y, count_in, count_out)
            writer.write(annotated)

            background     = self._update_background(background, frames_buffer,
                                                      gray.astype(np.float32))
            prev_centroids = centroids
            processed     += 1

        cap.release()
        writer.release()
        size_kb = os.path.getsize(out_path) / 1024
        print(f"  [{method_name:<15}] {os.path.basename(out_path):<30} "
              f"{processed:>5} frames  IN={count_in:>3}  OUT={count_out:>3}  ({size_kb:.0f} KB)")
        return out_path


# ═══════════════════════════════════════════════════════════════════════════════
#  SUBCLASS 1 — Mean Window
# ═══════════════════════════════════════════════════════════════════════════════

class MeanWindowBS(BackgroundSubtractionProcess):
    """Background = mean của N frame gần nhất (sliding window)."""

    def _init_background(self, cap):
        frames = []
        for _ in range(self.n_batch):
            ret, f = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32))
        return np.mean(frames, axis=0).astype(np.uint8), frames

    def _update_background(self, background, frames_buffer, new_gray):
        frames_buffer.pop(0)
        frames_buffer.append(new_gray)
        return np.mean(frames_buffer, axis=0).astype(np.uint8)

    def process(self):
        return self.run("MeanWindow")


# ═══════════════════════════════════════════════════════════════════════════════
#  SUBCLASS 2 — Median Window
# ═══════════════════════════════════════════════════════════════════════════════

class MedianWindowBS(BackgroundSubtractionProcess):
    """Background = median của N frame gần nhất — bền vững hơn mean với outlier."""

    def _init_background(self, cap):
        frames = []
        for _ in range(self.n_batch):
            ret, f = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32))
        return np.median(frames, axis=0).astype(np.uint8), frames

    def _update_background(self, background, frames_buffer, new_gray):
        frames_buffer.pop(0)
        frames_buffer.append(new_gray)
        return np.median(frames_buffer, axis=0).astype(np.uint8)

    def process(self):
        return self.run("MedianWindow")


# ═══════════════════════════════════════════════════════════════════════════════
#  SUBCLASS 3 — One Last Frame
# ═══════════════════════════════════════════════════════════════════════════════

class OneLastFrameBS(BackgroundSubtractionProcess):
    """Background = frame ngay trước (t-1). Đơn giản, nhạy với mọi chuyển động."""

    def _init_background(self, cap):
        ret, f = cap.read()
        gray   = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        return gray, None

    def _update_background(self, background, frames_buffer, new_gray):
        return new_gray.astype(np.uint8)

    def process(self):
        return self.run("OneLastFrame")


# ═══════════════════════════════════════════════════════════════════════════════
#  SUBCLASS 4 — MOG2  (dùng cv2.createBackgroundSubtractorMOG2)
# ═══════════════════════════════════════════════════════════════════════════════

class MOG2BS(BackgroundSubtractionProcess):
    """
    Background = Gaussian Mixture Model (cv2 built-in).
    Tự học nền qua từng frame, xử lý shadow (vùng tối = 127 → bỏ qua).
    """

    def __init__(self, *args, history=500, var_threshold=16, **kwargs):
        super().__init__(*args, **kwargs)
        self.subtractor = cv2.createBackgroundSubtractorMOG2(
            history=history, varThreshold=var_threshold, detectShadows=True)

    def _init_background(self, cap):
        return None, None   # MOG2 tự quản lý model nội bộ

    def _get_mask(self, frame, background):
        """Override: dùng MOG2 thay vì absdiff thủ công."""
        mask = self.subtractor.apply(frame)
        # Shadow (127) → loại bỏ, chỉ giữ foreground (255)
        _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)
        mask = cv2.dilate(mask, None, iterations=2)
        mask = cv2.erode(mask, None, iterations=2)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return mask, gray

    def process(self):
        return self.run("MOG2")


print("✓ Classes loaded: BackgroundSubtractionProcess, MeanWindowBS, MedianWindowBS, OneLastFrameBS, MOG2BS")

✓ Classes loaded: BackgroundSubtractionProcess, MeanWindowBS, MedianWindowBS, OneLastFrameBS, MOG2BS


In [13]:
# ── Chạy cả 4 phương pháp ─────────────────────────────────────────────────────
processors = [
    MeanWindowBS (path_to_video, n_batch=number_of_batch_frame, threshold=30, min_area=1000, line_ratio=0.5),
    MedianWindowBS(path_to_video, n_batch=number_of_batch_frame, threshold=30, min_area=1000, line_ratio=0.5),
    OneLastFrameBS(path_to_video, n_batch=number_of_batch_frame, threshold=30, min_area=1000, line_ratio=0.5),
    MOG2BS        (path_to_video, threshold=30, min_area=1000, line_ratio=0.5, history=500, var_threshold=16),
]

print("=" * 75)
print(f"  {'METHOD':<17} {'OUTPUT FILE':<32} {'FRAMES':>6}  {'IN':>4}  {'OUT':>4}  SIZE")
print("=" * 75)
outputs = [p.process() for p in processors]
print("=" * 75)
print(f"\nTất cả file đã lưu tại: {VIDEO_DIR}\\")

  METHOD            OUTPUT FILE                      FRAMES    IN   OUT  SIZE
  [MeanWindow     ] output_MeanWindow.mp4            491 frames  IN=  6  OUT=  8  (2595 KB)
  [MedianWindow   ] output_MedianWindow.mp4          491 frames  IN=  5  OUT=  7  (2495 KB)
  [OneLastFrame   ] output_OneLastFrame.mp4          496 frames  IN=  0  OUT=  0  (1645 KB)
  [MOG2           ] output_MOG2.mp4                  497 frames  IN=  4  OUT=  4  (2488 KB)

Tất cả file đã lưu tại: d:\Duy Toan\Python\DUT AI Club\Homework\day 28 3-3-2026\
